In [2]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

# 1. Problem Setup: Coordinates (x, y) on a 10x10 city map
np.random.seed(42)
NUM_RIDERS = 4
NUM_ORDERS = 6

riders = np.random.uniform(0, 10, size=(NUM_RIDERS, 2))  # Rider locations
orders = np.random.uniform(0, 10, size=(NUM_ORDERS, 2))  # Restaurant/Drop locations
max_orders_per_rider = 2  # Workload capacity limit

# 2. Fitness Function (Minimize Delivery Distance + Workload Penalty)
def evaluate_fitness(assignment):
    total_distance = 0.0
    rider_counts = np.zeros(NUM_RIDERS, dtype=int)

    for order_idx, rider_idx in enumerate(assignment):
        # Euclidean distance between assigned rider and order
        dist = np.linalg.norm(riders[rider_idx] - orders[order_idx])
        total_distance += dist
        rider_counts[rider_idx] += 1

    # Penalize if a rider exceeds max workload capacity
    overload = np.maximum(0, rider_counts - max_orders_per_rider)
    workload_penalty = np.sum(overload) * 15.0

    return total_distance + workload_penalty

# 3. Helper: Get von Neumann (4-neighbor) indices in a 2D grid
def get_neighbors(r, c, rows, cols):
    neighbors = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = (r + dr) % rows, (c + dc) % cols
        neighbors.append((nr, nc))
    return neighbors

# 4. Parallel Cellular Algorithm
def parallel_cellular_allocation(rows=4, cols=4, max_iter=50, crossover_rate=0.6, mutation_rate=0.1):
    # Initialize grid of cells (each cell holds an array of rider IDs for each order)
    grid = np.random.randint(0, NUM_RIDERS, size=(rows, cols, NUM_ORDERS))
    fitness_grid = np.zeros((rows, cols))

    global_best_assignment = None
    global_best_fitness = float('inf')

    with ThreadPoolExecutor() as executor:
        for iteration in range(max_iter):
            # Step A: Evaluate Fitness in Parallel
            cells = [grid[r, c] for r in range(rows) for c in range(cols)]
            results = list(executor.map(evaluate_fitness, cells))
            fitness_grid = np.array(results).reshape(rows, cols)

            # Update Global Best
            min_idx = np.unravel_index(np.argmin(fitness_grid), (rows, cols))
            if fitness_grid[min_idx] < global_best_fitness:
                global_best_fitness = fitness_grid[min_idx]
                global_best_assignment = grid[min_idx].copy()

            # Step B: Update States in Parallel using Neighbor Information
            def update_cell(pos):
                r, c = pos
                current_assign = grid[r, c].copy()
                best_neighbor_assign = current_assign
                best_neighbor_fit = fitness_grid[r, c]

                # Find best neighbor
                for nr, nc in get_neighbors(r, c, rows, cols):
                    if fitness_grid[nr, nc] < best_neighbor_fit:
                        best_neighbor_fit = fitness_grid[nr, nc]
                        best_neighbor_assign = grid[nr, nc]

                # Discrete Update Rule: Inherit assignments from best neighbor
                for o in range(NUM_ORDERS):
                    if np.random.rand() < crossover_rate:
                        current_assign[o] = best_neighbor_assign[o]
                    # Small random mutation to explore new rider options
                    if np.random.rand() < mutation_rate:
                        current_assign[o] = np.random.randint(0, NUM_RIDERS)

                return current_assign

            positions = [(r, c) for r in range(rows) for c in range(cols)]
            new_cells = list(executor.map(update_cell, positions))
            grid = np.array(new_cells).reshape(rows, cols, NUM_ORDERS)

    return global_best_assignment, global_best_fitness

# 5. Run Optimization and Output Results
best_assign, best_cost = parallel_cellular_allocation()

print(f"Optimal Allocation Cost (Distance + Penalty): {best_cost:.2f}\n")
for order_id, rider_id in enumerate(best_assign):
    dist = np.linalg.norm(riders[rider_id] - orders[order_id])
    print(f"Order {order_id} -> Assigned to Rider {rider_id} (Distance: {dist:.2f} km)")


Optimal Allocation Cost (Distance + Penalty): 14.46

Order 0 -> Assigned to Rider 1 (Distance: 1.71 km)
Order 1 -> Assigned to Rider 3 (Distance: 1.10 km)
Order 2 -> Assigned to Rider 1 (Distance: 3.99 km)
Order 3 -> Assigned to Rider 2 (Distance: 0.38 km)
Order 4 -> Assigned to Rider 3 (Distance: 4.21 km)
Order 5 -> Assigned to Rider 2 (Distance: 3.07 km)
